# Compare Stage 1 DPO and original SFT inference

This notebook compares the selected Stage 1 DPO adapter with its matching original SFT adapter on the same 160-question BioASQ dev split. Set MODEL_PRESET in the configuration cell to choose Qwen2.5-3B or Qwen2.5-0.5B; the default is **3B**.

For each model it generates **one shared bank of 10 samples per question**. The same samples are then scored in two ways:

1. **sample10_first5** — use only samples 1–5, remove invalid/duplicate answers while preserving their generation order, and submit up to five ranked answers.
2. **sample10_frequency_top5** — use all 10 samples, group answers with weak normalization, and rank unique answers by frequency; ties are broken by first occurrence.

An optional greedy single-answer baseline is also included. Frequency normalization changes case and repeated whitespace only; punctuation, hyphens, Greek letters, and biomedical surface markers remain significant.

Raw generations are saved after every question. Re-running the notebook with the same preset and run name resumes missing questions rather than regenerating completed ones.


In [ ]:
import csv
import gc
import hashlib
import json
import os
import re
import sys
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Any

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

import pandas as pd
import torch
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents, start / "Task-Structured Counterfactual Preference Mining"]:
        if (candidate / "cse_dpo").is_dir() and (candidate / "src").is_dir():
            return candidate.resolve()
    raise RuntimeError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cse_dpo.generated_bioasq_eval import load_gold_examples
from src.prompt_registry import resolve_prompt_bundle
from src.utility.bioasq_format import exact_answer_groups, parse_prediction_items
from src.utility.bioasq_official import evaluate_with_bioasq_java
from src.utility.config import QUESTION_INSTRUCTIONS
from src.utility.data import clean_text
from src.utility.eval_dataset import load_eval_examples, render_prompt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", PROJECT_ROOT)
print("Device:", torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else "CPU")

## Configuration

The default MODEL_PRESET="qwen25_3b" compares the original 3B SFT adapter against the Stage 1 checkpoint selected by dev MRR at step 32. Change it to "qwen25_05b" to reproduce the previous 0.5B comparison.

TEMPERATURE=0.7 and TOP_P=0.9 match the candidate-generation setting used elsewhere in this project. Raise TEMPERATURE only if you deliberately want a more diverse and noisier experiment. Keep the generated RUN_NAME unchanged to resume an interrupted run.


In [ ]:
MODEL_PRESET = "qwen25_3b"  # Choose "qwen25_3b" or "qwen25_05b".

MODEL_PRESETS = {
    "qwen25_05b": {
        "base_model": PROJECT_ROOT / "models/Qwen2.5-0.5B-Instruct",
        "sft_adapter": PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_05b_lora_dropout_005_strict_extractive/adapter_best_evidence_mrr",
        "stage1_adapter": PROJECT_ROOT / "Artifacts/cse_dpo/three_stage_dpo_qwen25_05b_c3_c1_curriculum_gold_supported_full_bp4096/stage_1_concept_learning/checkpoints/step_60",
        "stage1_model_name": "stage1_dpo_step60",
        "stage1_description": "Selected 0.5B Stage 1 concept-learning DPO checkpoint (step 60)",
        "run_name": "qwen25_05b_stage1_dpo_vs_original_sft_dev_sample10_t07_seed3407",
    },
    "qwen25_3b": {
        "base_model": PROJECT_ROOT / "models/Qwen2.5-3B-Instruct",
        "sft_adapter": PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_3b_lora_dropout_005_strict_extractive/adapter_best_evidence_mrr",
        "stage1_adapter": PROJECT_ROOT / "Artifacts/cse_dpo/three_stage_dpo_qwen25_3b_c3_c1_curriculum_semantic_full_bp2048/stage_1_concept_learning/checkpoints/step_32",
        "stage1_model_name": "stage1_dpo_step32",
        "stage1_description": "Selected 3B Stage 1 concept-learning DPO checkpoint (step 32, dev MRR 0.50625)",
        "run_name": "qwen25_3b_stage1_dpo_vs_original_sft_dev_sample10_t07_seed3407",
    },
}
if MODEL_PRESET not in MODEL_PRESETS:
    raise ValueError(f"Unknown MODEL_PRESET={MODEL_PRESET!r}; choose one of {sorted(MODEL_PRESETS)}")

PRESET = MODEL_PRESETS[MODEL_PRESET]
BASE_MODEL = PRESET["base_model"]
DPO_MODEL_NAME = PRESET["stage1_model_name"]
MODEL_CONFIGS = {
    "original_sft": {
        "adapter": PRESET["sft_adapter"],
        "description": f"Original strict-extractive {MODEL_PRESET} SFT selected by evidence MRR",
    },
    DPO_MODEL_NAME: {
        "adapter": PRESET["stage1_adapter"],
        "description": PRESET["stage1_description"],
    },
}

# Both 0.5B and 3B DPO evaluations used this same prepared 160-question dev file.
DEV_INPUT = PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b/eval_prepared.json"
PROMPT_REGISTRY = PROJECT_ROOT / "prompts/factoid_single_answer_aligned.json"
PROMPT_REF = "factoid-single-answer-extractive-v1"

SAMPLES_PER_QUESTION = 10
FIRST_N_SAMPLES = 5
MAX_RANKED_ANSWERS = 5
TEMPERATURE = 0.7
TOP_P = 0.9
SAMPLING_SEED = 3407
MAX_SEQ_LENGTH = 4096
MAX_NEW_TOKENS = 64
RUN_GREEDY_BASELINE = True
LIMIT_QUESTIONS = None  # Set to 3–5 for a smoke test, then restore to None.

# 4-bit loading lets each 3B model fit comfortably on a 16 GB GPU. Models are loaded
# and released sequentially; the two adapters are never resident at the same time.
LOAD_IN_4BIT = True
LOCAL_FILES_ONLY = True
USE_CACHE = False

RUN_NAME = PRESET["run_name"]
OUTPUT_DIR = PROJECT_ROOT / "Artifacts/cse_dpo/inference_strategy_comparisons" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, cfg in MODEL_CONFIGS.items():
    if not cfg["adapter"].exists():
        raise FileNotFoundError(f"{name} adapter does not exist: {cfg['adapter']}")
if not BASE_MODEL.exists():
    raise FileNotFoundError(BASE_MODEL)
if not DEV_INPUT.exists():
    raise FileNotFoundError(DEV_INPUT)

EXPERIMENT_CONFIG = {
    "model_preset": MODEL_PRESET,
    "base_model": str(BASE_MODEL.resolve()),
    "models": {name: {**cfg, "adapter": str(cfg["adapter"].resolve())} for name, cfg in MODEL_CONFIGS.items()},
    "dev_input": str(DEV_INPUT.resolve()),
    "prompt_ref": PROMPT_REF,
    "samples_per_question": SAMPLES_PER_QUESTION,
    "first_n_samples": FIRST_N_SAMPLES,
    "max_ranked_answers": MAX_RANKED_ANSWERS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "sampling_seed": SAMPLING_SEED,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "run_greedy_baseline": RUN_GREEDY_BASELINE,
    "limit_questions": LIMIT_QUESTIONS,
    "load_in_4bit": LOAD_IN_4BIT,
    "weak_frequency_normalization": "NFKC + casefold + whitespace collapse; punctuation preserved",
}
print(json.dumps({**EXPERIMENT_CONFIG, "output_dir": str(OUTPUT_DIR)}, indent=2, default=str))


In [ ]:
def source_args(limit=None):
    return type("SourceArgs", (), {
        "question_types": ["factoid"],
        "max_summary_answers": 5,
        "max_factoid_answers": 5,
        "max_list_items": 100,
        "max_resources": 0,
        "max_resource_chars": 0,
        "resource_selection": "first",
        "resource_granularity": "document",
        "resource_window_mode": "single",
        "resource_reranker_model": None,
        "resource_reranker_article_model": None,
        "resource_reranker_device": None,
        "resource_reranker_batch_size": 1,
        "local_files_only": True,
        "limit": limit,
    })()


prompt_bundle = resolve_prompt_bundle(PROMPT_REGISTRY, PROMPT_REF, QUESTION_INSTRUCTIONS)
examples = load_eval_examples([DEV_INPUT], source_args(LIMIT_QUESTIONS), prompt_bundle["instructions"])
gold_examples = load_gold_examples([DEV_INPUT])
examples = [x for x in examples if x.question_id in gold_examples]
if LIMIT_QUESTIONS is not None:
    examples = examples[: int(LIMIT_QUESTIONS)]
examples_by_id = {x.question_id: x for x in examples}

print("Prompt ID:", prompt_bundle["prompt_id"])
print("Dev questions:", len(examples))
assert len(examples) == (int(LIMIT_QUESTIONS) if LIMIT_QUESTIONS is not None else 160)


In [ ]:
def load_model_and_tokenizer(adapter_path: Path):
    tokenizer_path = adapter_path if (adapter_path / "tokenizer_config.json").exists() else BASE_MODEL
    tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path), local_files_only=LOCAL_FILES_ONLY)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    kwargs = {"low_cpu_mem_usage": True, "local_files_only": LOCAL_FILES_ONLY}
    if DEVICE.type == "cuda":
        kwargs["device_map"] = {"": DEVICE.index or 0}
    if LOAD_IN_4BIT and DEVICE.type == "cuda":
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    else:
        kwargs["torch_dtype"] = torch.float16 if DEVICE.type == "cuda" else torch.float32

    base = AutoModelForCausalLM.from_pretrained(str(BASE_MODEL), **kwargs)
    model = PeftModel.from_pretrained(base, str(adapter_path), local_files_only=LOCAL_FILES_ONLY)
    model.config.use_cache = USE_CACHE
    model.eval()
    return model, tokenizer


def release_model(model=None, tokenizer=None):
    if model is not None:
        model.to("cpu") if DEVICE.type == "cpu" else None
        del model
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def strip_answer_prefix(text: str) -> str:
    return re.sub(r"^answer\s*:\s*", "", clean_text(text), flags=re.IGNORECASE).strip()


def generate_one(model, tokenizer, prompt: str, *, do_sample: bool, seed: int | None = None) -> str:
    prior_side = getattr(tokenizer, "truncation_side", None)
    try:
        tokenizer.truncation_side = "left"
        encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    finally:
        if prior_side is not None:
            tokenizer.truncation_side = prior_side

    device = next(model.parameters()).device
    encoded = {k: v.to(device) for k, v in encoded.items()}
    kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "do_sample": do_sample,
        "use_cache": USE_CACHE,
    }
    if do_sample:
        kwargs.update({"temperature": TEMPERATURE, "top_p": TOP_P})
    if seed is not None:
        torch.manual_seed(int(seed))
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(int(seed))
    with torch.inference_mode():
        output_ids = model.generate(**encoded, **kwargs)
    prompt_len = encoded["input_ids"].shape[-1]
    text = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True)
    del encoded, output_ids
    return strip_answer_prefix(text)


## Generate and cache samples

This is the expensive cell. It loads one model at a time and appends one JSONL record after each completed question. If execution stops, run the cell again to continue.

In [ ]:
def write_json(path: Path, payload: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def read_jsonl_latest(path: Path) -> dict[str, dict[str, Any]]:
    latest = {}
    if path.exists():
        with path.open(encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    row = json.loads(line)
                    latest[row["question_id"]] = row
    return latest


def assert_resume_compatible(model_dir: Path, model_name: str):
    manifest_path = model_dir / "sampling_manifest.json"
    expected = {**EXPERIMENT_CONFIG, "model_name": model_name}
    if manifest_path.exists():
        previous = json.loads(manifest_path.read_text(encoding="utf-8"))
        for key in ["base_model", "dev_input", "prompt_ref", "samples_per_question", "temperature", "top_p", "sampling_seed", "max_seq_length", "max_new_tokens", "run_greedy_baseline", "limit_questions"]:
            if previous.get(key) != expected.get(key):
                raise ValueError(f"Cannot resume {model_name}: {key} changed ({previous.get(key)!r} -> {expected.get(key)!r}). Change RUN_NAME.")
        old_adapter = previous["models"][model_name]["adapter"]
        new_adapter = expected["models"][model_name]["adapter"]
        if old_adapter != new_adapter:
            raise ValueError(f"Cannot resume {model_name}: adapter changed. Change RUN_NAME.")
    else:
        write_json(manifest_path, expected)


def generate_model_bank(model_name: str, cfg: dict[str, Any]):
    model_dir = OUTPUT_DIR / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    assert_resume_compatible(model_dir, model_name)
    raw_path = model_dir / "raw_generations.jsonl"
    completed = read_jsonl_latest(raw_path)
    missing = [x for x in examples if x.question_id not in completed]
    print(f"{model_name}: {len(completed)} cached, {len(missing)} remaining")
    if not missing:
        return completed

    model, tokenizer = load_model_and_tokenizer(Path(cfg["adapter"]))
    try:
        with raw_path.open("a", encoding="utf-8") as out:
            for example in tqdm(missing, desc=f"Generate {model_name}"):
                row_index = next(i for i, x in enumerate(examples) if x.question_id == example.question_id)
                prompt = render_prompt(
                    tokenizer,
                    example,
                    chat_template=prompt_bundle.get("chat_template", "qwen-2.5"),
                    prompt_format="chat",
                )
                greedy = generate_one(model, tokenizer, prompt, do_sample=False) if RUN_GREEDY_BASELINE else None
                samples = [
                    generate_one(
                        model,
                        tokenizer,
                        prompt,
                        do_sample=True,
                        seed=SAMPLING_SEED + row_index * SAMPLES_PER_QUESTION + sample_index,
                    )
                    for sample_index in range(SAMPLES_PER_QUESTION)
                ]
                row = {
                    "question_id": example.question_id,
                    "question": example.body,
                    "greedy_generation": greedy,
                    "samples": samples,
                    "sample_seeds": [SAMPLING_SEED + row_index * SAMPLES_PER_QUESTION + i for i in range(SAMPLES_PER_QUESTION)],
                }
                out.write(json.dumps(row, ensure_ascii=False) + "\n")
                out.flush()
                completed[example.question_id] = row
    finally:
        release_model(model, tokenizer)
    return completed


banks = {}
for model_name, cfg in MODEL_CONFIGS.items():
    banks[model_name] = generate_model_bank(model_name, cfg)

print({name: len(rows) for name, rows in banks.items()})

In [ ]:
def extract_single_candidate(sample: str | None) -> str | None:
    if not sample:
        return None
    items = parse_prediction_items(sample, "factoid")
    if len(items) != 1:
        return None
    candidate = clean_text(items[0])
    return candidate or None


def weak_vote_key(candidate: str) -> str:
    # Deliberately preserve punctuation and hyphens.
    return " ".join(unicodedata.normalize("NFKC", candidate).casefold().split())


def rank_first_n_samples(samples: list[str], n: int = FIRST_N_SAMPLES) -> tuple[list[str], dict[str, Any]]:
    ranked, seen, invalid = [], set(), []
    for i, sample in enumerate(samples[:n]):
        candidate = extract_single_candidate(sample)
        if candidate is None:
            invalid.append({"sample_index": i, "sample": sample})
            continue
        key = weak_vote_key(candidate)
        if not key or key in seen:
            continue
        seen.add(key)
        ranked.append(candidate)
    return ranked[:MAX_RANKED_ANSWERS], {
        "samples_considered": min(n, len(samples)),
        "valid_unique_candidates": len(ranked),
        "invalid_samples": invalid,
    }


def rank_by_frequency(samples: list[str]) -> tuple[list[str], dict[str, Any]]:
    votes, representative, first_seen, invalid = Counter(), {}, {}, []
    for i, sample in enumerate(samples):
        candidate = extract_single_candidate(sample)
        if candidate is None:
            invalid.append({"sample_index": i, "sample": sample})
            continue
        key = weak_vote_key(candidate)
        if not key:
            invalid.append({"sample_index": i, "sample": sample})
            continue
        votes[key] += 1
        representative.setdefault(key, candidate)
        first_seen.setdefault(key, i)
    keys = sorted(votes, key=lambda key: (-votes[key], first_seen[key]))
    ranked = [representative[key] for key in keys[:MAX_RANKED_ANSWERS]]
    return ranked, {
        "samples_considered": len(samples),
        "valid_sample_count": int(sum(votes.values())),
        "unique_candidate_count": len(votes),
        "modal_frequency": max(votes.values(), default=0),
        "votes": {representative[key]: int(votes[key]) for key in keys},
        "invalid_samples": invalid,
    }


def format_prediction(candidates: list[str]) -> str:
    return " ".join(f"[BE] {clean_text(x)} [EE]" for x in candidates[:MAX_RANKED_ANSWERS] if clean_text(x))


def official_score_args():
    return type("OfficialArgs", (), {
        "bioasq_java_jar": str(PROJECT_ROOT / "third_party/Evaluation-Measures/flat/BioASQEvaluation/dist/BioASQEvaluation.jar"),
        "bioasq_java_heap": "512m",
        "bioasq_java_version": 5,
    })()


def evaluate_strategy(model_name: str, bank: dict[str, dict[str, Any]], strategy: str):
    strategy_dir = OUTPUT_DIR / model_name / strategy
    strategy_dir.mkdir(parents=True, exist_ok=True)
    rows=[]
    for example in examples:
        raw = bank[example.question_id]
        if strategy == "greedy_single":
            candidate = extract_single_candidate(raw.get("greedy_generation"))
            ranked = [candidate] if candidate else []
            diagnostics = {"raw_generation": raw.get("greedy_generation")}
        elif strategy == "sample10_first5":
            ranked, diagnostics = rank_first_n_samples(raw["samples"], FIRST_N_SAMPLES)
        elif strategy == "sample10_frequency_top5":
            ranked, diagnostics = rank_by_frequency(raw["samples"])
        else:
            raise ValueError(strategy)
        rows.append({
            "question_id": example.question_id,
            "question_type": example.question_type,
            "body": example.body,
            "source_path": example.source_path,
            "prediction": format_prediction(ranked),
            "ranked_candidates": ranked,
            "gold_output": example.gold_output,
            **diagnostics,
        })

    official = evaluate_with_bioasq_java(
        prediction_rows=rows,
        examples_by_key={(r["question_id"], r["question_type"]): gold_examples[r["question_id"]] for r in rows},
        model_label=f"{model_name}-{strategy}",
        model_dir=strategy_dir,
        args=official_score_args(),
        include_per_question=True,
    )
    by_id = {x["question_id"]: x for x in official["per_question"]}
    enriched=[]
    for row in rows:
        item=dict(row)
        item.update({k:v for k,v in by_id[row["question_id"]].items() if k not in {"question_id","question_type"}})
        enriched.append(item)
    metrics = official["aggregate"]["by_type"]["factoid"]["metrics"]
    summary = {
        "model": model_name,
        "strategy": strategy,
        "question_count": len(rows),
        "mrr": metrics["mrr"],
        "strict_accuracy": metrics["strict_accuracy"],
        "lenient_accuracy": metrics["lenient_accuracy"],
        "temperature": None if strategy == "greedy_single" else TEMPERATURE,
        "top_p": None if strategy == "greedy_single" else TOP_P,
        "samples_generated": 1 if strategy == "greedy_single" else SAMPLES_PER_QUESTION,
        "official_scores": str(strategy_dir / "official_bioasq/official_scores.json"),
    }
    write_json(strategy_dir / "summary.json", summary)
    write_json(strategy_dir / "per_question_metrics.json", enriched)
    pd.DataFrame(enriched).to_csv(strategy_dir / "per_question_metrics.csv", index=False)
    return summary, enriched


strategies = (["greedy_single"] if RUN_GREEDY_BASELINE else []) + ["sample10_first5", "sample10_frequency_top5"]
all_summaries=[]
all_results={}
for model_name, bank in banks.items():
    all_results[model_name]={}
    for strategy in strategies:
        summary, rows = evaluate_strategy(model_name, bank, strategy)
        all_summaries.append(summary)
        all_results[model_name][strategy]=rows
        print(json.dumps(summary, indent=2))

summary_df = pd.DataFrame(all_summaries).sort_values(["strategy", "model"]).reset_index(drop=True)
summary_df.to_csv(OUTPUT_DIR / "model_strategy_summary.csv", index=False)
write_json(OUTPUT_DIR / "model_strategy_summary.json", all_summaries)
display(summary_df)

In [ ]:
# Direct DPO-minus-SFT MRR difference for every inference strategy.
pivot = summary_df.pivot(index="strategy", columns="model", values="mrr")
if {"original_sft", DPO_MODEL_NAME}.issubset(pivot.columns):
    pivot["dpo_minus_sft_mrr"] = pivot[DPO_MODEL_NAME] - pivot["original_sft"]
pivot.to_csv(OUTPUT_DIR / "mrr_comparison_pivot.csv")
display(pivot)

# Question-level comparison makes gains and regressions auditable.
question_views=[]
for model_name in MODEL_CONFIGS:
    for strategy in strategies:
        df=pd.DataFrame(all_results[model_name][strategy])
        view=df[["question_id","body","prediction","ranked_candidates","mrr","strict_accuracy","lenient_accuracy"]].copy()
        prefix=f"{model_name}__{strategy}"
        view=view.rename(columns={c:f"{prefix}__{c}" for c in view.columns if c not in {"question_id","body"}})
        question_views.append(view)
comparison=question_views[0]
for view in question_views[1:]:
    comparison=comparison.merge(view,on=["question_id","body"],how="outer")
comparison.to_csv(OUTPUT_DIR / "question_level_comparison.csv",index=False)
print("Saved:", OUTPUT_DIR / "question_level_comparison.csv")
display(comparison.head())


## Interpretation notes

- `sample10_first5` estimates the result of directly submitting the first five stochastic candidates.
- `sample10_frequency_top5` estimates self-consistency voting. It can improve MRR when the correct answer recurs, but confidently repeated wrong answers can make it worse.
- Because both strategies are derived from the same ten samples, their difference comes only from aggregation rather than different random generations.
- The official Java evaluator determines MRR. The weak punctuation-preserving key is used only to group frequency votes and remove duplicates.